# Trabajo Práctico Integrador - Introducción al Análisis de Datos

**Caso:** análisis y predicción de cancelaciones en reservas hoteleras.

> TPI - Maestro


##Datos de identificación

- **Apellido y nombre:** Aguillón, Fernando
- **Comisión:** 5
- **Dataset asignado:** E
- **Entrega:** 1
- **Fecha:** 01/09/2026

##Importación de librerías

Importar solamente las librerías necesarias para la entrega que se está desarrollando.

In [ ]:
paquetes <- c("tidyverse", "lubridate")

instalar <- paquetes[!paquetes %in% rownames(installed.packages())]
if (length(instalar) > 0) {
  install.packages(instalar)
}

library(tidyverse)
library(lubridate)

##Carga del dataset

Cargar el archivo CSV correspondiente a la comisión. El notebook debe funcionar desde el dataset original, no desde objetos creados en otros notebooks.

In [ ]:
datos <- read.csv("/content/hotel booking TPI grupo E.csv", stringsAsFactors = FALSE)

##Presentación del problema

El presente trabajo plantea el análisis de un conjunto de datos correspondiente a reservas realizadas en hoteles urbanos y hoteles tipo resort. El objetivo principal es identificar patrones y características que puedan estar relacionados con la cancelación de las reservas.

El dataset contiene información sobre las reservas, incluyendo fechas de llegada, duración de las estadías, cantidad de huéspedes, canales de distribución, tipos de clientes, historial de cancelaciones y tarifas promedio por noche, entre otras variables.

A partir de esta información, se busca comprender el comportamiento de las reservas y estudiar qué características podrían estar asociadas con una mayor probabilidad de cancelación.

La variable principal del estudio será `is_canceled`, que indica si una reserva fue cancelada o no:

- `0`: reserva no cancelada.
- `1`: reserva cancelada.

El análisis se desarrollará mediante distintas etapas de preparación, exploración, análisis, visualización e interpretación de datos. En etapas posteriores también se buscará evaluar la posibilidad de construir un modelo predictivo que permita estimar la probabilidad de cancelación de una reserva.

## Relación entre datos, información y conocimiento

Para comprender el propósito del análisis, es importante diferenciar los conceptos de datos, información y conocimiento, ya que representan distintos niveles de aprovechamiento de los elementos disponibles.

**Datos:** son los valores individuales que se encuentran almacenados en el dataset, sin que necesariamente tengan una interpretación por sí solos. Por ejemplo, un registro puede contener un valor `1` en la variable `is_canceled`.

**Información:** surge cuando los datos se organizan, procesan y analizan para obtener resultados que permitan comprender una situación. Por ejemplo, al calcular la proporción de reservas canceladas se obtiene que aproximadamente el 37 % de las reservas del conjunto analizado fueron canceladas.

**Conocimiento:** se obtiene al interpretar la información y relacionarla con el problema que se busca resolver. Por ejemplo, si mediante análisis posteriores se descubre que las reservas realizadas con mayor anticipación presentan una proporción de cancelación diferente, este resultado podría aportar conocimiento útil para comprender el comportamiento de los clientes y orientar decisiones comerciales.

En este trabajo, el objetivo es transformar los datos disponibles en información relevante y, posteriormente, obtener conocimiento que permita comprender mejor los factores asociados a las cancelaciones hoteleras.

##Ciclo de vida del análisis de datos

El análisis de datos requiere un proceso organizado que permita pasar desde la identificación de un problema hasta la obtención de resultados que puedan utilizarse para tomar decisiones.

Para este trabajo se consideran las siguientes etapas:

**1. Identificación del problema:** consiste en comprender la situación que se busca analizar y establecer los objetivos del estudio. En este caso, se busca comprender y predecir las cancelaciones de reservas hoteleras.

**2. Obtención y comprensión de los datos:** consiste en identificar el origen de los datos, conocer su estructura y comprender qué representa cada variable. En esta primera entrega se realiza una exploración inicial del dataset asignado.

**3. Preparación de los datos:** comprende la revisión de valores faltantes, registros duplicados, inconsistencias y posibles valores atípicos. También incluye las transformaciones necesarias para que los datos puedan utilizarse correctamente.

**4. Análisis exploratorio:** consiste en aplicar técnicas estadísticas y herramientas de visualización para identificar distribuciones, comportamientos, patrones y posibles relaciones entre las variables.

**5. Modelado:** en esta etapa se pueden aplicar técnicas estadísticas y algoritmos de aprendizaje automático para construir modelos que permitan explicar o predecir determinados comportamientos. Para este trabajo, el interés principal será evaluar la posibilidad de predecir cancelaciones.

**6. Evaluación e interpretación:** consiste en comprobar la calidad de los resultados obtenidos, evaluar el desempeño de los modelos y determinar si las conclusiones responden a las preguntas iniciales.

**7. Comunicación de resultados:** finalmente, se presentan los hallazgos mediante explicaciones, gráficos y conclusiones que permitan comprender la información y utilizarla como apoyo para la toma de decisiones.

Estas etapas no necesariamente se desarrollan de manera estrictamente lineal. Durante el análisis pueden surgir problemas que requieran volver a etapas anteriores, corregir los datos o replantear determinadas decisiones.

## Desarrollo de la entrega 1

###Vista inicial
visualización de las primeras filas del dataset.

In [ ]:
head(datos)

###Estructura general
cantidad de filas, cantidad de columnas y nombres de las variables.

In [ ]:
dimensiones <- dim(datos)

cat("Cantidad de filas:", dimensiones[1], "\n")
cat("Cantidad de columnas:", dimensiones[2], "\n")
cat("Columnas del dataset:\n")
print(names(datos))

El dataset presenta las siguientes características, según la ejecución original:

- **Cantidad de registros:** 25.000.
- **Cantidad de variables:** 32.

Las variables identificadas son:

`booking_id`, `hotel`, `is_canceled`, `lead_time`, `arrival_date_year`, `arrival_date_month`, `arrival_date_week_number`, `arrival_date_day_of_month`, `arrival_date`, `stays_in_weekend_nights`, `stays_in_week_nights`, `adults`, `children`, `babies`, `meal`, `country`, `market_segment`, `distribution_channel`, `is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled`, `reserved_room_type`, `assigned_room_type`, `booking_changes`, `deposit_type`, `agent`, `company`, `days_in_waiting_list`, `customer_type`, `adr`, `required_car_parking_spaces` y `total_of_special_requests`.

### Tipos de datos y clasificación conceptual de las variables

Es importante diferenciar dos conceptos:

- **Clase de dato en R:** indica cómo se encuentra almacenado un valor dentro del entorno de programación. Algunos ejemplos son `integer`, `numeric` y `character`.
- **Tipo conceptual de variable:** representa la naturaleza de la información que contiene y permite clasificarla como numérica, categórica o temporal.

Por lo tanto, una variable almacenada como `integer` no necesariamente representa una cantidad numérica. Por ejemplo, `is_canceled` utiliza los valores 0 y 1, pero conceptualmente representa una categoría binaria.

In [ ]:
cat("\nClases de datos en R:\n")
print(sapply(datos, class))

A partir de las variables identificadas, se realiza la siguiente clasificación inicial:

| Variable | Clasificación conceptual |
|---|---|
| booking_id | Categórica (identificador) |
| hotel | Categórica nominal |
| is_canceled | Categórica binaria |
| lead_time | Numérica discreta |
| arrival_date_year | Temporal |
| arrival_date_month | Temporal |
| arrival_date_week_number | Temporal |
| arrival_date_day_of_month | Temporal |
| arrival_date | Temporal |
| stays_in_weekend_nights | Numérica discreta |
| stays_in_week_nights | Numérica discreta |
| adults | Numérica discreta |
| children | Numérica discreta |
| babies | Numérica discreta |
| meal | Categórica nominal |
| country | Categórica nominal |
| market_segment | Categórica nominal |
| distribution_channel | Categórica nominal |
| is_repeated_guest | Categórica binaria |
| previous_cancellations | Numérica discreta |
| previous_bookings_not_canceled | Numérica discreta |
| reserved_room_type | Categórica nominal |
| assigned_room_type | Categórica nominal |
| booking_changes | Numérica discreta |
| deposit_type | Categórica nominal |
| agent | Categórica (identificador) |
| company | Categórica (identificador) |
| days_in_waiting_list | Numérica discreta |
| customer_type | Categórica nominal |
| adr | Numérica continua |
| required_car_parking_spaces | Numérica discreta |
| total_of_special_requests | Numérica discreta |

Esta clasificación es inicial y podrá revisarse durante las siguientes etapas, especialmente cuando se realicen transformaciones o se analice con mayor profundidad el significado de cada variable.

En particular, las variables `agent` y `company` pueden encontrarse almacenadas como numéricas, pero sus valores representan identificadores y no cantidades sobre las que tenga sentido realizar operaciones aritméticas.

Por otro lado, `arrival_date` aparece almacenada como `character` en R, pero conceptualmente representa una fecha. Esto significa que, para realizar operaciones temporales correctamente, podría ser necesario convertirla al formato de fecha correspondiente.

###Caracterización descriptiva inicial

Se realiza una primera exploración estadística mediante la función `summary(datos)`, que permite obtener resúmenes generales de las variables y observar sus valores mínimos, máximos, medidas centrales y presencia de datos faltantes, según corresponda.

In [ ]:
summary(datos)

A partir de los resultados obtenidos se destacan las siguientes observaciones:

**Variable `is_canceled`**

La media de esta variable es de aproximadamente 0,3704. Debido a que utiliza los valores 0 y 1, este resultado representa una proporción de aproximadamente 37,04 % de reservas canceladas, mientras que el 62,96 % restante no presenta cancelaciones.

Esta variable resulta especialmente relevante porque constituye la variable objetivo del estudio.

**Variable `lead_time`**

Presenta una media aproximada de 104 días y una mediana de 69 días, con valores que van desde 0 hasta 629 días.

La diferencia entre la media y la mediana sugiere una distribución posiblemente asimétrica, en la que algunas reservas realizadas con mucha anticipación elevan el promedio.

**Duración de las estadías**

La variable `stays_in_week_nights` presenta una media aproximada de 2,5 noches y una mediana de 2 noches.

Por su parte, `stays_in_weekend_nights` presenta una media cercana a 0,9 noches.

Estos resultados permiten observar que, en general, las reservas corresponden a estadías relativamente cortas, aunque existen registros con duraciones considerablemente mayores.

**Cantidad de huéspedes**

La variable `adults` presenta una mediana de 2 adultos por reserva.

En cambio, la media de `children` es de aproximadamente 0,10 y la de `babies` es cercana a 0,007.

Esto indica que la mayoría de las reservas corresponden principalmente a adultos, mientras que la presencia de niños y bebés es considerablemente menor.

**Valores faltantes**

Se observa una presencia importante de datos faltantes en algunas variables. Entre ellas, `company` presenta una cantidad elevada de valores ausentes y `agent` también posee una cantidad significativa.

En cambio, variables como `hotel`, `meal`, `market_segment` y `deposit_type` no presentan valores ausentes según el resumen obtenido.

La presencia de estos valores deberá ser considerada antes de realizar análisis más profundos, ya que podría afectar los resultados y requerir un tratamiento específico.

**Variable `adr`**

La variable `adr`, correspondiente a la tarifa diaria promedio, presenta una mediana de 94,50 y una media aproximada de 102,06.

También se observa un valor mínimo de -6,38, que resulta llamativo para una tarifa hotelera. Este registro deberá ser revisado posteriormente para determinar si corresponde a un error, una situación particular o un problema de registro.

**Posibles valores extremos**

Algunas variables presentan valores máximos considerablemente superiores a sus valores centrales, como `stays_in_week_nights`, `stays_in_weekend_nights`, `days_in_waiting_list` y `previous_bookings_not_canceled`.

Estas diferencias podrían indicar la presencia de valores extremos que deberán analizarse antes de extraer conclusiones definitivas.

###Primeras observaciones y conclusiones

A partir de la exploración descriptiva inicial se observa que el conjunto de datos contiene 25.000 reservas hoteleras y 32 variables que incluyen información sobre las características de las reservas, las fechas de llegada, la duración de las estadías, los huéspedes y las cancelaciones.

Uno de los primeros resultados relevantes es que aproximadamente el 37 % de las reservas fueron canceladas. Este porcentaje permite dimensionar inicialmente el problema y justifica la importancia de estudiar qué características podrían estar relacionadas con las cancelaciones.

También se identifican diferencias entre las medias y medianas de algunas variables, como `lead_time` y `adr`, que sugieren posibles distribuciones asimétricas y justifican la necesidad de profundizar el análisis.

Por otro lado, la presencia de valores faltantes y posibles valores atípicos indica que será necesario realizar una etapa de preparación y limpieza antes de avanzar hacia análisis más complejos o modelos predictivos.

En esta primera etapa no se establecen relaciones causales ni se afirma que alguna variable sea responsable de las cancelaciones. Los resultados obtenidos representan únicamente una caracterización inicial del conjunto de datos.

###Consideraciones éticas iniciales

En esta primera etapa, los resultados deben interpretarse únicamente como una descripción general del conjunto de datos y no como evidencia de relaciones causales. La existencia de una determinada característica en las reservas no implica necesariamente que dicha característica sea la causa de una cancelación.

También es importante considerar la presencia de datos faltantes y posibles valores atípicos, ya que pueden afectar los resultados de los análisis posteriores. Cualquier tratamiento de estos datos deberá estar justificado y evitar modificaciones que puedan introducir sesgos.

El análisis debe realizarse respetando la privacidad de las personas involucradas. Aunque el conjunto de datos utilizado no presenta información personal directamente identificable entre las variables observadas, los datos de reservas deben utilizarse únicamente con fines analíticos legítimos y evitando intentar identificar o inferir información personal sobre individuos.

Además, si en etapas posteriores se construye un modelo predictivo, será importante evaluar sus posibles sesgos y limitaciones. Una predicción de cancelación no debería interpretarse como una certeza ni utilizarse automáticamente para tomar decisiones que perjudiquen a un cliente.

Por último, las conclusiones obtenidas deberán limitarse al conjunto de datos analizado y al contexto en el que fueron recopilados. No sería adecuado generalizar automáticamente estos resultados a todos los hoteles, huéspedes o reservas sin contar con evidencia adicional que permita sostener dicha generalización.

##Entrega 2

Esta segunda etapa continúa el trabajo iniciado en la Entrega 1. El objetivo es diagnosticar la calidad del conjunto de datos, justificar las decisiones de limpieza y preparar variables que puedan utilizarse en el análisis posterior de las cancelaciones hoteleras.

La limpieza no consiste en eliminar automáticamente todo valor inusual. Cada decisión debe considerar el significado de la variable y el problema que se desea estudiar. Por eso, se conservarán variables relevantes para explicar las cancelaciones y se registrará cada modificación en una bitácora.

**Criterios incorporados a partir de la devolución docente:**

- Se conservan `lead_time` y `stays_in_week_nights`.
- Se incorporan variables derivadas.
- Se documentan las transformaciones en una bitácora.
- Se revisa `adr` distinguiendo valores negativos, ceros y valores extremos, y se informa la cantidad exacta de filas afectadas mediante el propio código.


###Carga del dataset original

In [ ]:
ruta_dataset <- "/content/hotel booking TPI grupo E.csv"

df_original <- read.csv(
  ruta_dataset,
  stringsAsFactors = FALSE,
  na.strings = c("", "NA")
)

df_hoteles <- df_original

cat("Dimensiones originales:", nrow(df_hoteles), "filas y",
    ncol(df_hoteles), "columnas\\n")
stopifnot("is_canceled" %in% names(df_hoteles))

###Diagnóstico inicial

Antes de modificar los datos, se conserva una copia del dataset original y se revisan dimensiones, estructura, duplicados y valores faltantes. Los resultados se calculan al ejecutar el notebook, evitando consignar cantidades que no hayan sido verificadas.

In [ ]:
dim(df_hoteles)
str(df_hoteles)
summary(df_hoteles)

cat("Filas duplicadas completas:", sum(duplicated(df_hoteles)), "\\n")

contar_faltantes <- function(x) {
  if (is.character(x)) {
    sum(is.na(x) | trimws(x) == "")
  } else {
    sum(is.na(x))
  }
}

faltantes_iniciales <- sapply(df_hoteles, contar_faltantes)
sort(faltantes_iniciales[faltantes_iniciales > 0], decreasing = TRUE)

###Bitácora de limpieza y transformación
La bitácora registra el problema detectado, la variable afectada, la decisión, su justificación y la cantidad de registros antes y después de la operación. Así se puede auditar el proceso y comprobar que las explicaciones coincidan con los resultados.

In [ ]:
bitacora <- data.frame(
  etapa = character(),
  problema = character(),
  variable = character(),
  decision = character(),
  justificacion = character(),
  filas_antes = integer(),
  filas_despues = integer(),
  filas_afectadas = integer(),
  stringsAsFactors = FALSE
)

registrar <- function(etapa, problema, variable, decision,
                      justificacion, antes, despues) {
  bitacora[nrow(bitacora) + 1, ] <<- list(
    etapa, problema, variable, decision, justificacion,
    as.integer(antes), as.integer(despues),
    as.integer(antes - despues)
  )
}

### Registros duplicados

Se identifican duplicados completos. Solo se eliminan las filas que sean idénticas en todas sus columnas. No se eliminan reservas que compartan atributos, porque pueden ser observaciones distintas.

In [ ]:
filas_antes <- nrow(df_hoteles)
duplicados <- sum(duplicated(df_hoteles))

if (duplicados > 0) {
  df_hoteles <- df_hoteles[!duplicated(df_hoteles), ]
}

registrar(
  "Limpieza", "Registros completamente duplicados", "Todas las variables",
  "Eliminar duplicados exactos",
  "Una fila idéntica en todas sus columnas constituye una repetición del mismo registro.",
  filas_antes, nrow(df_hoteles)
)

cat("Duplicados detectados y eliminados:", duplicados, "\\n")

### Selección de variables

Se elimina `booking_id` porque es un identificador único y no describe una característica de la reserva. Se eliminan las columnas de componentes de fecha (`arrival_date_year`, `arrival_date_month`, `arrival_date_week_number` y `arrival_date_day_of_month`) porque se conservará `arrival_date` y se derivarán de ella el año y el mes.

Se conservan `lead_time` y `stays_in_week_nights`: el primero mide la anticipación de la reserva y puede relacionarse con la cancelación; el segundo permite conocer la duración de la estadía y construir `total_nights`. También se conservan `meal` y `distribution_channel`, ya que son características potencialmente relevantes para estudiar el comportamiento de las reservas.

In [ ]:
columnas_eliminar <- intersect(
  c("booking_id", "arrival_date_year", "arrival_date_month",
    "arrival_date_week_number", "arrival_date_day_of_month"),
  names(df_hoteles)
)

filas_antes <- nrow(df_hoteles)
columnas_antes <- ncol(df_hoteles)
df_hoteles <- df_hoteles[, !(names(df_hoteles) %in% columnas_eliminar), drop = FALSE]

registrar(
  "Selección", "Identificador y componentes redundantes de fecha",
  paste(columnas_eliminar, collapse = ", "),
  "Eliminar columnas indicadas",
  "Se conserva arrival_date como fecha completa; lead_time, stays_in_week_nights, meal y distribution_channel permanecen por su posible valor explicativo.",
  filas_antes, nrow(df_hoteles)
)

cat("Columnas eliminadas:", paste(columnas_eliminar, collapse = ", "), "\\n")
cat("Columnas antes:", columnas_antes,
    "| columnas después:", ncol(df_hoteles), "\\n")

### Valores faltantes

Se revisan los faltantes por variable y se toman decisiones considerando su significado:

- `company`: se elimina porque aproximadamente el 94,2 % de sus registros no tiene dato. La variable quedaría sustentada por una proporción muy pequeña de observaciones.
- `agent`: se conserva. Aunque presenta faltantes, el código de agente puede representar un canal de intermediación útil. Se transforma a texto y los faltantes se identifican como `Sin agente`, sin tratarlos como una cantidad numérica.
- `country`: los faltantes se reemplazan por `Sin especificar`, conservando las filas y haciendo explícita la ausencia de información.
- `children`: los dos faltantes detectados en el diagnóstico inicial se reemplazan por cero, decisión que debe interpretarse como ausencia de niños informados. Se registra expresamente para que pueda revisarse si el contexto del dato lo requiere.

La bitácora y los conteos permiten comprobar cuántas celdas se modificaron y si se eliminaron filas o columnas.

In [ ]:
# Eliminar company por su proporción extremadamente alta de faltantes
if ("company" %in% names(df_hoteles)) {
  n_company_na <- contar_faltantes(df_hoteles$company)
  prop_company_na <- n_company_na / nrow(df_hoteles)

  filas_antes <- nrow(df_hoteles)
  df_hoteles$company <- NULL

  registrar(
    "Faltantes", "Proporción muy elevada de datos faltantes",
    "company", "Eliminar variable",
    paste0("Faltantes: ", n_company_na, " (",
           round(prop_company_na * 100, 2),
           "%). La cobertura resulta insuficiente para un análisis confiable."),
    filas_antes, nrow(df_hoteles)
  )
}

# Conservar agent como identificador categórico y explicitar los faltantes
if ("agent" %in% names(df_hoteles)) {
  n_agent_na <- contar_faltantes(df_hoteles$agent)
  df_hoteles$agent <- as.character(df_hoteles$agent)
  df_hoteles$agent[is.na(df_hoteles$agent) |
                     trimws(df_hoteles$agent) == ""] <- "Sin agente"

  registrar(
    "Faltantes", "Códigos de agente ausentes",
    "agent", "Convertir a categórica y asignar categoría Sin agente",
    "El código identifica agentes y no es una magnitud numérica; se conserva la variable y se hace explícita la ausencia.",
    nrow(df_hoteles), nrow(df_hoteles)
  )
}

if ("country" %in% names(df_hoteles)) {
  n_country_na <- contar_faltantes(df_hoteles$country)
  df_hoteles$country[is.na(df_hoteles$country) |
                       trimws(df_hoteles$country) == ""] <- "Sin especificar"

  registrar(
    "Faltantes", "País de origen ausente",
    "country", "Reemplazar faltantes por Sin especificar",
    "Se preservan las reservas y se distingue la ausencia de información de una categoría real.",
    nrow(df_hoteles), nrow(df_hoteles)
  )
}

if ("children" %in% names(df_hoteles)) {
  n_children_na <- sum(is.na(df_hoteles$children))
  df_hoteles$children[is.na(df_hoteles$children)] <- 0

  registrar(
    "Faltantes", "Cantidad de niños ausente",
    "children", "Reemplazar NA por 0",
    "En el diagnóstico inicial se observaron dos faltantes. Se adopta cero como ausencia de niños informados y se deja documentada la imputación.",
    nrow(df_hoteles), nrow(df_hoteles)
  )
}

cat("Faltantes restantes por variable:\\n")
faltantes_finales_limpieza <- sapply(df_hoteles, contar_faltantes)
print(faltantes_finales_limpieza[faltantes_finales_limpieza > 0])

### Revisión de valores de `adr`

`adr` representa la tarifa diaria promedio. Se diferencian tres situaciones:

- **Valores negativos:** no son coherentes con una tarifa diaria promedio y se consideran inválidos.
- **Valores iguales a cero:** no se eliminan automáticamente. Pueden representar una tarifa sin cargo, una condición particular o un registro que requiere análisis contextual.
- **Valores altos:** se investigan como posibles valores extremos, pero no se eliminan solo por superar un umbral. Un valor atípico no necesariamente es un error.

Para corregir la inconsistencia señalada por el docente, el código calcula y muestra la cantidad exacta de registros con `adr` negativo y la cantidad de filas eliminadas. La explicación no fija una cifra manual: el resultado ejecutado es la referencia.

In [ ]:
if ("adr" %in% names(df_hoteles)) {
  cat("Resumen de adr antes del tratamiento:\\n")
  print(summary(df_hoteles$adr))

  cantidad_negativos <- sum(df_hoteles$adr < 0, na.rm = TRUE)
  cantidad_ceros <- sum(df_hoteles$adr == 0, na.rm = TRUE)
  cantidad_na_adr <- sum(is.na(df_hoteles$adr))

  filas_antes <- nrow(df_hoteles)
  df_hoteles <- df_hoteles[
    is.na(df_hoteles$adr) | df_hoteles$adr >= 0,
    , drop = FALSE
  ]
  filas_despues <- nrow(df_hoteles)

  registrar(
    "Valores inválidos", "Tarifas diarias promedio negativas",
    "adr", "Eliminar únicamente filas con adr < 0",
    "Una tarifa promedio negativa no tiene interpretación válida en este contexto. Los ceros se conservan porque pueden responder a situaciones reales que requieren análisis.",
    filas_antes, filas_despues
  )

  cat("Registros con adr negativo detectados:", cantidad_negativos, "\\n")
  cat("Registros con adr igual a cero conservados:", cantidad_ceros, "\\n")
  cat("Valores faltantes de adr:", cantidad_na_adr, "\\n")
  cat("Filas eliminadas por adr negativo:", filas_antes - filas_despues, "\\n")
  cat("Filas restantes:", filas_despues, "\\n")
} else {
  warning("No se encontró la variable adr.")
}

### Reservas sin huéspedes informados

Se identifica la cantidad de reservas en las que `adults`, `children` y `babies` son simultáneamente cero. No se eliminan automáticamente: una reserva cancelada o modificada podría presentar esta combinación y eliminarla podría sesgar el análisis de `is_canceled`.

En su lugar, se crea una variable indicadora para poder estudiar estos casos y decidir más adelante, con evidencia, si corresponde excluirlos.

In [ ]:
if (all(c("adults", "children", "babies") %in% names(df_hoteles))) {
  df_hoteles$reserva_sin_huespedes <-
    df_hoteles$adults == 0 &
    df_hoteles$children == 0 &
    df_hoteles$babies == 0

  cat("Reservas sin huéspedes informados:",
      sum(df_hoteles$reserva_sin_huespedes, na.rm = TRUE), "\\n")
}

### Detección de valores atípicos

Se utiliza el criterio del rango intercuartílico (IQR): un valor se marca como atípico si está por debajo de Q1 − 1,5 × IQR o por encima de Q3 + 1,5 × IQR.

La detección no implica eliminación. Variables como `lead_time`, duración de estadía, cancelaciones previas y `adr` pueden presentar distribuciones asimétricas y valores altos reales. Se informa la cantidad de observaciones señaladas para su revisión.

In [ ]:
detectar_outliers <- function(x) {
  x <- x[is.finite(x)]
  if (length(x) < 4) return(integer(0))

  q1 <- quantile(x, 0.25, na.rm = TRUE, names = FALSE)
  q3 <- quantile(x, 0.75, na.rm = TRUE, names = FALSE)
  iqr <- q3 - q1

  which(x < (q1 - 1.5 * iqr) | x > (q3 + 1.5 * iqr))
}

variables_numericas_revision <- intersect(
  c("lead_time", "stays_in_weekend_nights", "stays_in_week_nights",
    "previous_cancellations", "previous_bookings_not_canceled",
    "booking_changes", "days_in_waiting_list", "adr"),
  names(df_hoteles)
)

resumen_outliers <- lapply(variables_numericas_revision, function(v) {
  x <- df_hoteles[[v]]
  x_valido <- x[!is.na(x)]
  q1 <- quantile(x_valido, 0.25, na.rm = TRUE)
  q3 <- quantile(x_valido, 0.75, na.rm = TRUE)
  iqr <- q3 - q1
  data.frame(
    variable = v,
    cantidad_atipicos = sum(x_valido < (q1 - 1.5 * iqr) |
                            x_valido > (q3 + 1.5 * iqr)),
    limite_inferior = as.numeric(q1 - 1.5 * iqr),
    limite_superior = as.numeric(q3 + 1.5 * iqr)
  )
})

resumen_outliers <- bind_rows(resumen_outliers)
print(resumen_outliers)

cat("Los valores atípicos se conservan en esta etapa; no se eliminan sin una justificación contextual.\\n")

###Transformación y creación de variables derivadas

Las variables derivadas permiten representar características que no estaban expresadas directamente en una única columna. Se conservan las variables originales cuando siguen siendo útiles para interpretar los resultados.

###Fecha de llegada: conversión y componentes

Se convierte `arrival_date` al tipo fecha y se derivan el año y el mes. Esto permite realizar agrupaciones temporales sin depender de columnas redundantes del archivo original.

In [ ]:
if ("arrival_date" %in% names(df_hoteles)) {
  fecha_convertida <- as.Date(df_hoteles$arrival_date, format = "%Y-%m-%d")

  cat("Fechas que no pudieron convertirse:",
      sum(is.na(fecha_convertida) & !is.na(df_hoteles$arrival_date)), "\\n")

  df_hoteles$arrival_date <- fecha_convertida
  df_hoteles$arrival_year <- year(df_hoteles$arrival_date)
  df_hoteles$arrival_month <- month(df_hoteles$arrival_date,
                                    label = TRUE, abbr = FALSE)
}

### Duración total de la estadía

Se crea `total_nights`, sumando las noches de semana y de fin de semana. Esta variable facilita estudiar la duración completa de cada reserva. Se mantienen las dos variables originales para permitir análisis diferenciados.

In [ ]:
if (all(c("stays_in_week_nights", "stays_in_weekend_nights") %in%
          names(df_hoteles))) {
  df_hoteles$total_nights <-
    df_hoteles$stays_in_week_nights +
    df_hoteles$stays_in_weekend_nights
}

### Cantidad total de huéspedes

Se crea `total_guests`, sumando adultos, niños y bebés. La variable representa el tamaño total del grupo asociado a la reserva.

In [ ]:
if (all(c("adults", "children", "babies") %in% names(df_hoteles))) {
  df_hoteles$total_guests <-
    df_hoteles$adults + df_hoteles$children + df_hoteles$babies
}

###Indicador de reserva anticipada

Se crea `early_booking`, una variable binaria que indica si la reserva se realizó con más de 90 días de anticipación. El umbral de 90 días se adopta como una segmentación operativa para explorar la relación entre anticipación y cancelación; no implica que ese límite sea una regla universal.

In [ ]:
if ("lead_time" %in% names(df_hoteles)) {
  df_hoteles$early_booking <- ifelse(
    is.na(df_hoteles$lead_time), NA,
    ifelse(df_hoteles$lead_time > 90, 1L, 0L)
  )
}

###Estandarización de `adr`

Se crea `adr_standard`, que expresa la tarifa en unidades de desvío estándar respecto de la media. Se conserva `adr` original para mantener la interpretación en su unidad de medida.

La estandarización no elimina valores atípicos ni corrige errores: solo cambia la escala.

if ("adr" %in% names(df_hoteles)) {
  media_adr <- mean(df_hoteles$adr, na.rm = TRUE)
  desvio_adr <- sd(df_hoteles$adr, na.rm = TRUE)

  if (is.finite(desvio_adr) && desvio_adr > 0) {
    df_hoteles$adr_standard <- (df_hoteles$adr - media_adr) / desvio_adr
  } else {
    warning("No se pudo estandarizar adr porque el desvío es cero o no está definido.")
  }
}

### Codificación binaria de la variable `hotel`

Se generan variables indicadoras (0/1) para las categorías de hotel. Se conserva la variable categórica original, porque es más fácil de interpretar y puede ser útil en tablas y gráficos. Las columnas binarias pueden facilitar el uso posterior de determinados modelos.

In [ ]:
if ("hotel" %in% names(df_hoteles)) {
  niveles_hotel <- sort(unique(na.omit(df_hoteles$hotel)))

  for (nivel in niveles_hotel) {
    nombre_columna <- paste0("hotel_", make.names(nivel))
    df_hoteles[[nombre_columna]] <- as.integer(df_hoteles$hotel == nivel)
  }
}

### Registro de las transformaciones

Se incorporan a la bitácora las variables derivadas y las transformaciones realizadas. En esta sección también se verifica que las variables originales relevantes continúen presentes.

In [ ]:
registrar(
  "Transformación", "Representación temporal",
  "arrival_date, arrival_year, arrival_month",
  "Convertir fecha y derivar año y mes",
  "Permite analizar la distribución temporal de las llegadas y conserva la fecha completa.",
  nrow(df_hoteles), nrow(df_hoteles)
)

registrar(
  "Transformación", "Duración total de la reserva",
  "total_nights",
  "Sumar noches de semana y fin de semana",
  "Resume la duración total de la estadía; se conservan las variables que componen la suma.",
  nrow(df_hoteles), nrow(df_hoteles)
)

registrar(
  "Transformación", "Tamaño del grupo",
  "total_guests",
  "Sumar adultos, niños y bebés",
  "Permite estudiar el tamaño total del grupo de huéspedes.",
  nrow(df_hoteles), nrow(df_hoteles)
)

registrar(
  "Transformación", "Anticipación de la reserva",
  "early_booking",
  "Crear indicador: lead_time > 90",
  "Segmenta las reservas según un umbral operativo de anticipación.",
  nrow(df_hoteles), nrow(df_hoteles)
)

registrar(
  "Transformación", "Escala de tarifa",
  "adr_standard",
  "Estandarizar adr mediante media y desvío estándar",
  "Permite comparar la tarifa en una escala común; se conserva adr original.",
  nrow(df_hoteles), nrow(df_hoteles)
)

registrar(
  "Transformación", "Codificación de tipo de hotel",
  "hotel_*",
  "Crear indicadores binarios por categoría",
  "Facilita la utilización de la variable categórica en modelos; se conserva hotel original.",
  nrow(df_hoteles), nrow(df_hoteles)
)

print(bitacora)

###Verificación del conjunto preparado

Se revisan las dimensiones finales, las variables disponibles, los faltantes y los resúmenes estadísticos. Esta comprobación permite detectar errores introducidos durante la limpieza o transformación.

In [ ]:
cat("Dimensiones finales:", nrow(df_hoteles), "filas y",
    ncol(df_hoteles), "columnas\\n")

cat("\\nVariables finales:\\n")
print(names(df_hoteles))

cat("\\nFaltantes finales por variable:\\n")
faltantes_finales <- sapply(df_hoteles, contar_faltantes)
print(faltantes_finales[faltantes_finales > 0])

cat("\\nResumen de variables principales:\\n")
variables_resumen <- intersect(
  c("is_canceled", "lead_time", "stays_in_week_nights",
    "stays_in_weekend_nights", "total_nights", "total_guests",
    "adr", "adr_standard"),
  names(df_hoteles)
)
print(summary(df_hoteles[, variables_resumen, drop = FALSE]))

cat("\\nDistribución de la variable objetivo is_canceled:\\n")
print(table(df_hoteles$is_canceled, useNA = "ifany"))
print(prop.table(table(df_hoteles$is_canceled, useNA = "no")))

###Controles de conservación de variables relevantes

Se comprueba explícitamente que las variables señaladas por el docente no hayan sido eliminadas.

In [ ]:
variables_que_deben_conservarse <- c(
  "lead_time", "stays_in_week_nights", "is_canceled"
)

variables_ausentes <- setdiff(variables_que_deben_conservarse,
                              names(df_hoteles))

if (length(variables_ausentes) == 0) {
  cat("Control correcto: se conservaron lead_time, stays_in_week_nights e is_canceled.\\n")
} else {
  stop(paste("Faltan variables requeridas:",
             paste(variables_ausentes, collapse = ", ")))
}

variables_derivadas_esperadas <- c(
  "total_nights", "total_guests", "early_booking",
  "adr_standard", "arrival_year", "arrival_month"
)

cat("Variables derivadas presentes:\\n")
print(intersect(variables_derivadas_esperadas, names(df_hoteles)))

###Bitácora final y resumen de modificaciones

La bitácora completa permite consultar cada decisión y sus motivos. Las filas afectadas se calculan a partir de las dimensiones antes y después de cada operación; en las transformaciones que no eliminan filas, el valor es cero. Para los reemplazos de valores faltantes se informa la cantidad detectada en las salidas específicas de cada sección.

In [ ]:
print(bitacora, row.names = FALSE)

cat("\\nResumen de filas por etapa:\\n")
print(bitacora %>%
        group_by(etapa) %>%
        summarise(
          filas_afectadas = sum(filas_afectadas),
          operaciones_registradas = n(),
          .groups = "drop"
        ))

cat("\\nFilas originales:", nrow(df_original), "\\n")
cat("Filas finales:", nrow(df_hoteles), "\\n")
cat("Diferencia neta de filas:", nrow(df_original) - nrow(df_hoteles), "\\n")

###Conclusiones

En esta entrega se realizó un proceso de diagnóstico, limpieza y transformación del dataset de reservas hoteleras. Las decisiones se tomaron considerando el significado de las variables y su posible relación con la cancelación de reservas.

A diferencia de una limpieza basada únicamente en eliminar columnas o valores inusuales, se conservaron variables relevantes como `lead_time` y `stays_in_week_nights`. También se mantuvieron `meal`, `distribution_channel` y `agent`, ya que pueden aportar información para caracterizar las reservas. Se eliminó `booking_id` por su función identificadora y se reemplazaron las columnas de componentes de fecha por una fecha completa, de la que luego se derivaron año y mes.

En el tratamiento de faltantes se eliminó `company` debido a su proporción muy elevada de valores ausentes. Se conservaron otras variables con faltantes mediante categorías explícitas o imputación documentada. En `adr` se eliminaron exclusivamente los valores negativos, se conservaron los ceros y se revisaron los valores extremos sin eliminarlos automáticamente. La cantidad exacta de registros eliminados se obtiene de la ejecución del código y queda reflejada en la bitácora.

Finalmente, se crearon variables derivadas como `total_nights`, `total_guests`, `early_booking`, `adr_standard`, `arrival_year` y `arrival_month`, además de indicadores binarios para el tipo de hotel. Estas transformaciones amplían las posibilidades de análisis sin perder las variables originales necesarias para interpretar los resultados.

El conjunto resultante queda preparado para continuar con el análisis exploratorio y estudiar la relación entre las características de las reservas y la variable objetivo `is_canceled`.

**Importante:** los resultados numéricos definitivos deben tomarse de la ejecución completa de este notebook sobre el CSV original. No se fijan manualmente cantidades de filas eliminadas ni estadísticas que puedan variar según el archivo cargado.
